# Streamlit Application — Quant Research Intelligence System

**Pipeline:** FAISS + BM25 → RRF → Cross-Encoder → Gemini RAG → Citations / Traceability

Graph retrieval and caching are intentionally excluded to keep the application simple.


In [ ]:
import os
import re
import json
import pickle
from pathlib import Path

import streamlit as st
import numpy as np
import faiss
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer, CrossEncoder
from google import genai

# Works when Streamlit is launched from the project root or from notebooks/
PROJECT_ROOT = Path(__file__).resolve().parent.parent if "__file__" in globals() and Path(__file__).resolve().parent.name == "notebooks" else Path.cwd()
if not (PROJECT_ROOT / "data" / "vector_store").exists():
    PROJECT_ROOT = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()

VECTOR_STORE = PROJECT_ROOT / "data" / "vector_store"
load_dotenv(PROJECT_ROOT / ".env", override=True)

EMBEDDING_MODEL_NAME = "BAAI/bge-small-en-v1.5"
RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
LLM_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.5-flash-lite")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    st.error("GEMINI_API_KEY is missing from the project .env file.")
    st.stop()


In [ ]:
# Load models and retrieval artifacts once when the application starts.
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
reranker = CrossEncoder(RERANKER_MODEL_NAME)

faiss_index = faiss.read_index(str(VECTOR_STORE / "quant_research.faiss"))

with open(VECTOR_STORE / "quant_research_metadata.json", "r", encoding="utf-8") as f:
    metadata = json.load(f)

with open(VECTOR_STORE / "quant_research_bm25.pkl", "rb") as f:
    bm25 = pickle.load(f)

with open(VECTOR_STORE / "quant_research_bm25_ids.json", "r", encoding="utf-8") as f:
    bm25_ids = json.load(f)

chunk_lookup = {item["chunk_id"]: item for item in metadata}
client = genai.Client(api_key=GEMINI_API_KEY)


In [ ]:
def rrf_fusion(dense_ids, lexical_ids, k=60):
    scores = {}

    for rank, chunk_id in enumerate(dense_ids, start=1):
        scores[chunk_id] = scores.get(chunk_id, 0.0) + 1 / (k + rank)

    for rank, chunk_id in enumerate(lexical_ids, start=1):
        scores[chunk_id] = scores.get(chunk_id, 0.0) + 1 / (k + rank)

    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


def retrieve_and_rerank(query, dense_k=10, bm25_k=10, final_k=5):
    # 1. FAISS dense retrieval
    query_vector = embedding_model.encode(
        [query], normalize_embeddings=True, convert_to_numpy=True
    ).astype("float32")

    _, dense_indices = faiss_index.search(query_vector, dense_k)
    dense_ids = [metadata[i]["chunk_id"] for i in dense_indices[0] if i >= 0]

    # 2. BM25 lexical retrieval
    query_tokens = re.findall(r"\b\w+\b", query.lower())
    bm25_scores = bm25.get_scores(query_tokens)
    bm25_order = np.argsort(bm25_scores)[::-1][:bm25_k]
    lexical_ids = [bm25_ids[i] for i in bm25_order]

    # 3. Reciprocal Rank Fusion
    fused = rrf_fusion(dense_ids, lexical_ids)
    hybrid_ids = [chunk_id for chunk_id, _ in fused[:10]]

    # 4. Cross-encoder reranking
    pairs = [
        (query, chunk_lookup[chunk_id].get("retrieval_text", chunk_lookup[chunk_id].get("text", "")))
        for chunk_id in hybrid_ids
    ]
    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(hybrid_ids, scores),
        key=lambda x: float(x[1]),
        reverse=True,
    )[:final_k]

    return [chunk_id for chunk_id, _ in ranked]


In [ ]:
SYSTEM_PROMPT = """
You are a quantitative research assistant.
Answer only from the supplied research context.
Do not use outside knowledge.
If the context does not contain enough information, say so.
Cite substantive claims using [S1], [S2], etc.
"""


def check_input_guardrail(query):
    patterns = [
        r"ignore (all |the )?(previous|prior) instructions",
        r"reveal (the )?system prompt",
        r"show (me )?(the )?api key",
        r"jailbreak",
    ]
    blocked = any(re.search(p, query.lower()) for p in patterns)
    return {"blocked": blocked}


def generate_grounded_answer(query, context):
    guardrail = check_input_guardrail(query)

    if guardrail["blocked"]:
        return {
            "status": "blocked",
            "answer": "I can't process that instruction because it attempts to override the system instructions or request protected information.",
            "citations": [],
            "guardrail": guardrail,
        }

    prompt = f"""
CONTEXT:
{context}

QUESTION:
{query}

Answer only from CONTEXT and use [S1], [S2], ... citations.
"""

    response = client.models.generate_content(
        model=LLM_MODEL,
        contents=prompt,
        config=genai.types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            temperature=0.1,
            max_output_tokens=900,
        ),
    )

    answer = (response.text or "").strip()
    cited_numbers = sorted({int(x) for x in re.findall(r"\[S(\d+)\]", answer)})

    return {
        "status": "success",
        "answer": answer,
        "citations": [f"S{x}" for x in cited_numbers],
        "guardrail": guardrail,
    }


In [ ]:
def ask_question(query):
    if not query.strip():
        return {"status": "error", "answer": "Please enter a question.", "citations": []}

    reranked_ids = retrieve_and_rerank(query)

    context_parts = []
    source_records = []

    for rank, chunk_id in enumerate(reranked_ids, start=1):
        item = chunk_lookup[chunk_id]
        context_parts.append(
            f"[S{rank}] {item.get('retrieval_text', item.get('text', ''))}"
        )
        source_records.append({
            "citation": f"S{rank}",
            "chunk_id": chunk_id,
            "source_file": item.get("source_file"),
            "pages": item.get("pages", []),
            "section_path": item.get("section_path", []),
            "element_ids": item.get("element_ids", []),
            "figure_ids": item.get("figure_ids", []),
            "table_ids": item.get("table_ids", []),
            "formula_ids": item.get("formula_ids", []),
        })

    context = "\n\n".join(context_parts)
    result = generate_grounded_answer(query, context)

    result["traceability"] = [
        record for record in source_records
        if record["citation"] in result.get("citations", [])
    ]
    result["retrieved_chunks"] = reranked_ids
    return result


In [ ]:
st.set_page_config(
    page_title="Quant Research Intelligence System",
    page_icon="📚",
    layout="wide",
)

st.title("Quant Research Intelligence System")
st.write("Ask questions about quantitative finance research papers.")

query = st.text_input("Ask a research question")

if st.button("Search", type="primary"):
    with st.spinner("Retrieving papers and generating answer..."):
        result = ask_question(query)

    if result["status"] == "blocked":
        st.warning(result["answer"])
    elif result["status"] == "error":
        st.warning(result["answer"])
    else:
        st.subheader("Answer")
        st.markdown(result["answer"])

        st.subheader("Sources / Traceability")
        if result.get("traceability"):
            for source in result["traceability"]:
                pages = ", ".join(map(str, source.get("pages", []))) or "N/A"
                section = " → ".join(source.get("section_path", [])) or "N/A"
                st.write(
                    f"{source['citation']} — {source.get('source_file', 'Unknown')} | "
                    f"Pages: {pages} | Section: {section} | Chunk: {source['chunk_id']}"
                )
        else:
            st.write("No cited sources returned.")

with st.sidebar:
    st.title("About")
    st.write("Quant Research Intelligence System")
    st.subheader("Pipeline")
    st.write("FAISS → BM25 → RRF → Cross-Encoder → Gemini RAG → Citations")

    st.subheader("Example Questions")
    st.write("- What is stochastic volatility?")
    st.write("- Explain option pricing.")
    st.write("- Compare volatility models.")


### Run

Save this notebook inside the project's `notebooks` folder, convert it to Python, then run the generated script with Streamlit.

```powershell
python -m nbconvert --to script notebooks\11_streamlit_app.ipynb
streamlit run notebooks\11_streamlit_app.py
```
